In [ ]:
!pip install transformers
!pip install accelerate

In [ ]:
import pandas as pd

In [ ]:
import gdown
mental_health_data_link = 'https://drive.google.com/uc?export=view&id=11CGO0DV-E3EpeRo8cdfoE41TK63JO5fE'
gdown.download(mental_health_data_link, quiet=False)

In [ ]:
df = pd.read_csv("mental_health_data.csv", dtype = str)

In [ ]:
df.dropna(axis = 0, inplace = True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
# Transform Context and Response so that all white space characters are converted to space
df["Context"] = df["Context"].apply(lambda x: " ".join(x.split()))
df["Response"] = df["Response"].apply(lambda x: " ".join(x.split()))

In [ ]:
patient_tag = "<|patient|>"
doctor_tag = "<|therapist|>"
df["patient_query_response"] = patient_tag + "\n" + df["Context"] + "\n" + doctor_tag + "\n" + df["Response"] + "<|endoftext|>\n"

In [ ]:
print(df["patient_query_response"].iloc[0])

In [ ]:
num_examples = df.shape[0]
num_training_examples = int(num_examples * 0.95)
num_test_examples = num_examples - num_training_examples

In [ ]:
training_set = df.patient_query_response.values[0: num_training_examples]
test_set = df.patient_query_response.values[num_training_examples:]

In [ ]:
print(len(training_set))
print(len(test_set))

In [ ]:
with open('training_set.txt','w') as f:
  f.write(''.join(training_set))
with open('test_set.txt','w') as f:
  f.write(''.join(test_set))

In [ ]:
!head -20 training_set.txt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')
model = AutoModelForCausalLM.from_pretrained('distilgpt2')

In [ ]:
input_text = 'I fell from my bicycle and I broke my'
enc_input = tokenizer.encode(input_text,return_tensors='pt', add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 70,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)

In [ ]:
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')

In [ ]:
input_text = test_set[0].split(doctor_tag)[0]  +  doctor_tag + "\n"
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 150,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)

In [ ]:
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')

In [ ]:
#!curl https://webpages.scu.edu/ftp/msamorani/NLP/run_lm_finetuning.py > run_lm_finetuning.py
fixed_fine_tuning_code = 'https://drive.google.com/uc?export=view&id=1wJyT5k29A9tkpQV0OITdLHOpddFYjW4j'
gdown.download(fixed_fine_tuning_code, quiet=False)

In [ ]:
!mkdir experiments

In [ ]:
epochs=105
file_with_training_set = 'training_set.txt'

In [ ]:
# this is bash code to fine-tune a language model. File from here: https://github.com/alontalmor/pytorch-transformers/blob/master/examples/run_lm_finetuning.py
text = f"for epoch in {epochs} \n"+\
"do \n"+\
"python run_lm_finetuning.py "+\
f"--output_dir=experiments/epoch_{epochs} "+\
"--model_type=gpt2 "+\
"--model_name_or_path=distilgpt2 "+\
f"--train_data_file={file_with_training_set} "+\
"--do_train "+\
"--overwrite_output_dir "+\
"--save_steps=5000 " +\
f"--num_train_epochs={epochs} \n" +\
"done"

In [ ]:
f = open('run_experiments.sh',mode='w')
f.write(text)
f.close()

In [ ]:
!bash run_experiments.sh

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f'experiments/epoch_{epochs}')
model = AutoModelForCausalLM.from_pretrained(f'experiments/epoch_{epochs}')

model_path = f"mental_health_model_epoch_{epochs}"
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
# mount it
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
# copy it as a new directory in the root of your google drive
import shutil
shutil.copytree(model_path,'/content/drive/MyDrive/'+ model_path)

In [ ]:
input_text = test_set[0].split(doctor_tag)[0]  +  doctor_tag + "\n"
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 150,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')


In [ ]:
input_text = 'Today I fell from my bicycle and broke my '
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    max_length= 70,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')

In [ ]:
input_text = "Patient:\nWhy is everything hard?\nDoctor:\n"
enc_input = tokenizer.encode(input_text,return_tensors='pt',add_special_tokens=False)
output_sequences = model.generate(
    input_ids = enc_input,
    pad_token_id=tokenizer.eos_token_id,
    max_length= 150,  # the length of the final sentence
    temperature = 0.9, # the closer to one, the less deterministic. The closer to zero, the more deterministic
    top_k = 20, # how many next words to consider when doing a tree-like structure
    top_p = 0.9,
    repetition_penalty = 1, # penalty for repeating a word in the input (min 1)
    do_sample = True, # True -> probabilistic model (output varies)
    num_return_sequences = 5 # number of output sentences
)
for i in range(len(output_sequences)):
  print(f'{i}: {tokenizer.decode(output_sequences[i])}\n')
